# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdeenMir/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Classification.

Why: You're predicting binary outcomes (clicked or not clicked, relevant or not relevant, anomaly or normal). Classification is the right task type for yes/no or categorical predictions. Given your computer vision background, you could also frame ranking problems as pairwise classification, but for this internship, start with binary classification.

In [6]:
task_type = "Classification (binary)"
reason = "Predicting click/no-click or relevant/not-relevant outcomes from content features"
print(f"Task: {task_type}")
print(f"Reason: {reason}")


Task: Classification (binary)
Reason: Predicting click/no-click or relevant/not-relevant outcomes from content features


## 2. Target or proxy

Target: Whether a piece of content gets clicked / engaged with by users.

Where it comes from: Observed outcome — actual user behavior from search logs or content refresh logs. This is ground truth, not a rule you defined.

Why this matters: You're not inventing a label. Users already told you what they think by clicking or not clicking. That's the signal.

In [7]:
target_variable = "clicked"  # or "engagement" or "user_action"
source = "observed"  # not synthetic, not rule-based
label_distribution = {"clicked": 0.35, "not_clicked": 0.65}  # example

print(f"Predicting: {target_variable}")
print(f"Source: {source} outcome from user behavior")
print(f"Label split (example): {label_distribution}")

Predicting: clicked
Source: observed outcome from user behavior
Label split (example): {'clicked': 0.35, 'not_clicked': 0.65}


## 3. Success metric

Primary metric: F1-score (or ROC-AUC).

Why F1: In ranking/content systems, you care about both precision (don't show bad content) and recall (don't miss good content). F1 balances both. AUC is also valid if you want to evaluate across all thresholds.

Minimum acceptable: 0.70 F1 (if your baseline random guess is ~0.50, beating it by 40% is credible).

In [8]:
from sklearn.metrics import f1_score, roc_auc_score

metric = "F1-score"
threshold_good = 0.70
rationale = "Balances precision (avoid false positives) and recall (don't miss true positives)"

print(f"Primary metric: {metric}")
print(f"Good performance: > {threshold_good}")
print(f"Why: {rationale}")

Primary metric: F1-score
Good performance: > 0.7
Why: Balances precision (avoid false positives) and recall (don't miss true positives)


## 4. The unit of analysis, as a real dataframe

One row = one content item (or one search result / query-result pair).

Each row has:

Features (what we know about the content: title, domain, recency, metadata, etc.)
Target (whether users clicked it: 0 or 1)

In [9]:
import pandas as pd

# Create example dataframe matching your lane
unit_analysis = pd.DataFrame({
    'content_id': [1001, 1002, 1003, 1004, 1005],
    'query': ['python tutorial', 'python tutorial', 'javascript', 'javascript', 'react'],
    'title': ['Learn Python Basics', 'Advanced Python', 'JS Fundamentals', 'Modern JS', 'React 101'],
    'domain': ['example.com', 'tutorial.com', 'mdn.com', 'javascript.info', 'react.dev'],
    'position': [1, 2, 1, 2, 1],
    'recency_days': [5, 30, 2, 15, 1],
    'content_length': [2500, 5000, 1800, 3200, 2100],
    'clicked': [1, 0, 1, 1, 0]  # target
})

print(f"Unit of analysis: One row = one content item")
print(f"Total records: {len(unit_analysis)}")
print(unit_analysis)
print(f"\nTarget distribution:")
print(unit_analysis['clicked'].value_counts())

Unit of analysis: One row = one content item
Total records: 5
   content_id            query                title           domain  \
0        1001  python tutorial  Learn Python Basics      example.com   
1        1002  python tutorial      Advanced Python     tutorial.com   
2        1003       javascript      JS Fundamentals          mdn.com   
3        1004       javascript            Modern JS  javascript.info   
4        1005            react            React 101        react.dev   

   position  recency_days  content_length  clicked  
0         1             5            2500        1  
1         2            30            5000        0  
2         1             2            1800        1  
3         2            15            3200        1  
4         1             1            2100        0  

Target distribution:
clicked
1    3
0    2
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

Because the signal is noisy and multi-dimensional.

A fixed rule like "if position == 1, then predict click" gets ~65% right (from your earlier data). But it misses patterns:

Some position-2 results get clicked more than position-1 (if they're more relevant)
Domain reputation matters (trusted domains get clicked even if ranked lower)
Query type matters (navigational queries have different click patterns than informational)
Recency matters (fresh content gets more clicks)
Title length/relevance matters (better-written titles get clicks)

A single if-statement can't combine all these signals. ML learns the interactions between features — when to weight domain over position, when recency matters more, etc. That's why a decision tree or logistic regression beats a hand-coded rule.

In [10]:
# Demonstrate why a simple rule fails
import pandas as pd

# Example data
data = {
    'position': [1, 1, 2, 2, 3, 3],
    'domain_trusted': [1, 0, 1, 0, 1, 0],
    'recency_days': [5, 30, 2, 1, 15, 60],
    'clicked': [1, 0, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

# Simple rule: if position == 1, predict click
df['simple_rule_prediction'] = (df['position'] == 1).astype(int)

# Score
simple_accuracy = (df['simple_rule_prediction'] == df['clicked']).mean()

print(f"Simple rule accuracy (position == 1): {simple_accuracy:.2%}")
print(f"\nBut look at row 2: position=2, trusted_domain=1, recency=2 days → clicked=1")
print(f"The rule missed it because it only looks at position.")
print(f"\nML can learn: position + domain + recency → better predictions")
print(f"\nWhy ML wins: Combines multiple signals, learns their interactions, adapts to context.")

Simple rule accuracy (position == 1): 50.00%

But look at row 2: position=2, trusted_domain=1, recency=2 days → clicked=1
The rule missed it because it only looks at position.

ML can learn: position + domain + recency → better predictions

Why ML wins: Combines multiple signals, learns their interactions, adapts to context.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.